# Post-Vote Small Shareholder Trading Regressions

In [1]:
import stata_setup

stata_setup.config('/Applications/StataNow', 'mp')


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      StataNow 19.5
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2025 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user 2-core network, expiring 31 Oct 2026
Serial number: 501909305069
  Licensed to: Yichen Luo
               UCL

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


In [2]:
%%stata

clear all
set more off
set varabbrev off
version 18

* Paths
global PROCESSED_DATA "/Users/yichenluo/Library/CloudStorage/Dropbox/dao-governance/processed_data"
global TABLES "tables"
cap mkdir "$TABLES"


. 
. clear all

. set more off

. set varabbrev off

. version 18

. 
. * Paths
. global PROCESSED_DATA "/Users/yichenluo/Library/CloudStorage/Dropbox/dao-gove
> rnance/processed_data"

. global TABLES "tables"

. cap mkdir "$TABLES"

. 


In [3]:
%%stata

* Load regression-ready panel built by scripts/reg/reg_post_vote_trading.py
import delimited using "$PROCESSED_DATA/post_vote_trading_panel.csv", clear

* Parse date and fixed effects
gen day = date(date, "YMD")
format day %td
gen month = month(day)
gen quarter = quarter(day)
gen year = year(day)
encode gecko_id, gen(token)
encode space,    gen(dao)
encode topic,    gen(proposal_type)

* Dependent and independent variables
local dep bought sold
local treatment non_whale_victory_vp non_whale_victory_vn non_whale_victory_vp_vn against_outcome

capture confirm variable against_outcome
if _rc {
    gen against_outcome = vote_against_outcome
}

* Label variables
label var bought          "\${\it Buy}_{i,j,t}\$"
label var sold            "\${\it Sell}_{i,j,t}\$"
label var small_victory   "\${\it Small\ Victory}_{j,t}\$"
label var non_whale_victory_vp    "\${\it Small\ Victory}^{VP}_{j,t}\$"
label var non_whale_victory_vn    "\${\it Small\ Victory}^{VN}_{j,t}\$"
label var non_whale_victory_vp_vn "\${\it Small\ Victory}^{VP,VN}_{j,t}\$"
label var against_outcome "\${\it Against\ Outcome}_{i,j,t}\$"
label var log_vp          "\${\it log}(VP_{i,j,t}+1)\$"
label var weighted        "\${\it Weighted}_{j,t}\$"
label var quadratic       "\${\it Quadratic Voting}_{j,t}\$"
label var ranked_choice   "\${\it Ranked Choice}_{j,t}\$"
label var quorum          "\${\it Quorum}_{j,t}\$"
label var multi_choices   "\${\it Multiple Choices}_{j,t}\$"
label var have_discussion "\${\it Discussion}_{j,t}\$"
label var delegation      "\${\it Delegation}_{j,t}\$"
label var professionalism "\${\it Professionalism}_{j,t}\$"
label var concensus       "\${\it Concensus}_{j,t}\$"


. 
. * Load regression-ready panel built by scripts/reg/reg_post_vote_trading.py
. import delimited using "$PROCESSED_DATA/post_vote_trading_panel.csv", clear
(encoding automatically selected: ISO-8859-1)
(51 vars, 190,013 obs)

. 
. * Parse date and fixed effects
. gen day = date(date, "YMD")

. format day %td

. gen month = month(day)

. gen quarter = quarter(day)

. gen year = year(day)

. encode gecko_id, gen(token)

. encode space,    gen(dao)

. encode topic,    gen(proposal_type)

. 
. * Dependent and independent variables
. local dep bought sold

. local treatment non_whale_victory_vp non_whale_victory_vn non_whale_victory_v
> p_vn against_outcome

. 
. capture confirm variable against_outcome

. if _rc {
.     gen against_outcome = vote_against_outcome
. }

. 
. * Label variables
. label var bought          "\${\it Buy}_{i,j,t}\$"

. label var sold            "\${\it Sell}_{i,j,t}\$"

. label var small_victory   "\${\it Small\ Victory}_{j,t}\$"

. label var non_whale_victory_

## Simple PPML Buy/Sell Relationships

In [4]:
%%stata

******************************************************
* SIMPLE PPML HDFE MODELS
******************************************************

eststo clear

foreach d of local dep {
    ppmlhdfe `d' non_whale_victory_vp, absorb(year voter) vce(cluster voter)
    eststo `d'_vp
    estadd local fe_type "Y"
    estadd local fe_time "Y"

    ppmlhdfe `d' non_whale_victory_vn, absorb(year voter) vce(cluster voter)
    eststo `d'_vn
    estadd local fe_type "Y"
    estadd local fe_time "Y"

    ppmlhdfe `d' non_whale_victory_vp_vn, absorb(year voter) vce(cluster voter)
    eststo `d'_both
    estadd local fe_type "Y"
    estadd local fe_time "Y"

    ppmlhdfe `d' against_outcome, absorb(year proposal_type) vce(cluster proposal_type)
    eststo `d'_against
    estadd local fe_type "Y"
    estadd local fe_time "Y"
}

* Export LaTeX table
esttab                                                           ///
    bought_vp bought_vn bought_both bought_against              ///
    sold_vp sold_vn sold_both sold_against                      ///
    using "$TABLES/reg_post_vote_trading.tex", replace           ///
    se star(* 0.10 ** 0.05 *** 0.01) label nogaps nocon nonumbers ///
    nonotes booktabs                                             ///
    b(%9.3f) se(%9.2f)                                           ///
    keep(non_whale_victory_vp non_whale_victory_vn non_whale_victory_vp_vn against_outcome) ///
    mgroups("\${\it Buy}_{i,j,t}\$"                          ///
            "\${\it Sell}_{i,j,t}\$",                        ///
            span                                                 ///
            pattern(1 0 0 0 1 0 0 0)                             ///
            prefix(\multicolumn{@span}{c}{)                      ///
            suffix(})                                            ///
            erepeat(\cmidrule(lr){@span}) )                      ///
    mtitles("\${\it Small Win}^{vp}_{i,t}\$" "\${\it Small Win}^{vn}_{i,t}\$" "\${\it Small Win}^{vp,vn}_{i,t}\$" "Against Outcome" ///
            "\${\it Small Win}^{vp}_{i,t}\$" "\${\it Small Win}^{vn}_{i,t}\$" "\${\it Small Win}^{vp,vn}_{i,t}\$" "Against Outcome") ///
    posthead("\cmidrule(lr){2-5}\cmidrule(lr){6-9} &\multicolumn{1}{c}{(1)}&\multicolumn{1}{c}{(2)}&\multicolumn{1}{c}{(3)}&\multicolumn{1}{c}{(4)}&\multicolumn{1}{c}{(5)}&\multicolumn{1}{c}{(6)}&\multicolumn{1}{c}{(7)}&\multicolumn{1}{c}{(8)}\\\midrule") ///
    substitute("\_" "_")                                      ///
    stats(fe_type fe_time N r2_p,                                ///
        fmt(0 0 %9.0fc %9.3f)                                   ///
        labels("Voter FE" "Year FE" "Observations" "Pseudo R²"))


. 
. ******************************************************
. * SIMPLE PPML HDFE MODELS
. ******************************************************
. 
. eststo clear

. 
. foreach d of local dep {
  2.     ppmlhdfe `d' non_whale_victory_vp, absorb(year voter) vce(cluster vote
> r)
  3.     eststo `d'_vp
  4.     estadd local fe_type "Y"
  5.     estadd local fe_time "Y"
  6. 
.     ppmlhdfe `d' non_whale_victory_vn, absorb(year voter) vce(cluster voter)
  7.     eststo `d'_vn
  8.     estadd local fe_type "Y"
  9.     estadd local fe_time "Y"
 10. 
.     ppmlhdfe `d' non_whale_victory_vp_vn, absorb(year voter) vce(cluster vote
> r)
 11.     eststo `d'_both
 12.     estadd local fe_type "Y"
 13.     estadd local fe_time "Y"
 14. 
.     ppmlhdfe `d' against_outcome, absorb(year proposal_type) vce(cluster prop
> osal_type)
 15.     eststo `d'_against
 16.     estadd local fe_type "Y"
 17.     estadd local fe_time "Y"
 18. }
(dropped 161911 observations that are either singletons or separate